# Assignment 1 — Working with Open Data

In this assignment you take a **real, public dataset**, ask questions of it, answer them, and document your work so that someone else could reproduce it.

**A few things to know before you start:**
- This course is about **data, not code.** The point of A1 is the analytical questions and what the numbers tell you about how people behave — *not* whether you remember Python syntax.
- You can do the analysis **in Python here, or in Google Sheets.** Every question below works either way. Pick whatever lets you think about the data instead of fighting the tool.
- You may use an **AI assistant** (ChatGPT, Claude, …) to help you write code — with rules. See *A note on using AI* at the bottom before you do.

**What you'll turn in** (committed to your `hcde-410` repo):
1. This notebook, with your answers filled in.
2. A short **README.md** describing your work.
3. A **LICENSE** file, and a note about the dataset's own license.

Your commit history is part of the record — commit as you go.

## Part 0 — Get the data

The data comes from [data.seattle.gov](https://data.seattle.gov/), the City of Seattle's open-data portal. We'll use the [Burke-Gilman Trail counter north of NE 70th St](https://data.seattle.gov/Transportation/Burke-Gilman-Trail-north-of-NE-70th-St-Bicycle-and/2z5v-ecg8), which records **bike and pedestrian traffic, by hour, by direction**.

We'll pull the data straight from the city's API rather than downloading a file by hand. That's a small reproducibility win: anyone who runs this notebook gets the same data the same way, and it's written down exactly where it came from.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# The dataset's API endpoint, and a query for one year of data (2019).
# You can paste the full URL into a browser to preview the raw data.
api_endpoint = "https://data.seattle.gov/resource/2z5v-ecg8.json?"
api_parameters = "$limit=50000&$where=date > '2019-01-01T00:00:00' AND date < '2020-01-01T00:00:00'"

response = requests.get(api_endpoint + api_parameters)
raw_data = response.json()
print("Downloaded", len(raw_data), "hourly records.")
# If this cell fails (e.g. the API is down), see the note below the next cell.

Now load it into a **pandas DataFrame** — a table, like a spreadsheet you can compute with.

In [ ]:
df = pd.DataFrame(raw_data)
df.head()

### Quick sanity checks

Whenever you work with data **you didn't create**, check that it means what you think before you analyze it. Two things need fixing here:
- the counts arrive as **text**, so we convert them to numbers;
- the `date` is a string, so we convert it to a real **datetime** and pull out the hour and weekday.

We'll also add tidy `bikes` and `peds` columns (north + south).

In [ ]:
# convert the count columns from text to numbers
count_cols = ["bike_north", "bike_south", "ped_north", "ped_south", "bgt_north_of_ne_70th_total"]
df[count_cols] = df[count_cols].apply(pd.to_numeric, errors="coerce")

# convert the date string to a real datetime, and derive hour + weekday
df["date"] = pd.to_datetime(df["date"])
df["hour"] = df["date"].dt.hour
df["weekday"] = df["date"].dt.day_name()

# combined bike and pedestrian counts
df["bikes"] = df["bike_north"] + df["bike_south"]
df["peds"]  = df["ped_north"] + df["ped_south"]

df.info()
df.describe()

**Look at what `describe()` tells you.** Are there hours with 0 traffic? Any missing values (a sensor can go offline)? Note anything odd — you'll want to mention data limitations later.

**No-code option:** run the next cell to save the cleaned data as a CSV, then open it in Google Sheets and do your analysis there. If you go this route, title your sheet **"A1 — [your name]"**, share it so anyone with the link can view, and paste the link into your written answers.

In [ ]:
df.to_csv("bg_hourly_2019.csv", index=False)
print("Saved bg_hourly_2019.csv — a row per hour of 2019.")

## Part 1 — One worked example: when do cyclists ride?

Here is the whole technique, **once**, so you can see the pattern. After this, the questions are yours.

The move is always the same: **group → summarize → plot → interpret.** In pandas that's a couple of lines (no loops needed).

In [ ]:
# group every row by its hour of day, and sum bikes and peds within each hour
by_hour = df.groupby("hour")[["bikes", "peds"]].sum()

by_hour.plot(figsize=(11, 5))
plt.title("Burke-Gilman traffic by hour of day (2019)")
plt.xlabel("Hour of day (0–23)")
plt.ylabel("Total count in 2019")
plt.xticks(range(0, 24))
plt.show()

**Now read the plot — this is the actual work.** What does the shape tell you? Do you see one peak or two? What might a double peak mean about *why* people are on the trail? That interpretation — turning a shape into a claim about human behavior — is what every question below is really asking for.

## Part 2 — Your analysis

Answer each question below. For every one:
1. **Show how you got there** — a pandas cell here, or (no-code) a link to your Google Sheet in the answer.
2. **Interpret it in 2–4 sentences** — what does this say about how people actually use the trail? The interpretation is the point, not the number.

There isn't a single right answer to most of these. Making a reasonable choice and *defending* it is the skill.

**Q1. When is the trail busiest, and when is it quietest?**
Decide what *busy* means to you first — a single hour? a stretch of the day? bikes and peds together, or separately? — and say why you chose that.

In [ ]:
# Your code for Q1 (or put a Google Sheets link in your written answer below)
hourly_total = (
    df.assign(total=df["bikes"] + df["peds"])
    .groupby("hour")["total"]
    .sum()
)

busiest_hour = hourly_total.idxmax()# idxmax finds the hour that have most people, max finds how many people in that hour.

print(f"Busiest hour: {busiest_hour:02d}:00–{busiest_hour:02d}:59")

**Your Q1 answer:**

I define “busy” as the hour of the day with the highest combined bike and ped traffic. I chose this definition because a higher number of passages indicates heavier use of the trail during that time. I first added the bikes and peds counts, then summed them by hour of the day across 2019 and identified the highest total. The busiest time was 17:00–17:59. And the quietest hour is midnight(23:00-4:00) where there is not much people using the trail.

**Q2. Do cyclists and pedestrians use the trail at different times of day?**
Compare their hourly patterns. If they differ, what might explain it?

In [ ]:
# Your code for Q2
by_hour

**Your Q2 answer:**

Cyclists and pedestrians follow broadly similar daily patterns, with slightly different peak times and low traffic late at night. Bicycle traffic has a smaller peak at 8:00 and its highest peak at 17:00, while pedestrian traffic peaks at 9:00 and has a smaller peak at 16:00. The bicycle peaks align with typical morning and evening commuting times, possibly reflecting trips to and from work or school, while the pedestrian morning peak may reflect walking or exercise. These are possible explanations, since the data does not directly record why people use the trail. I asked the AI for help and chose to use by_hour to display the numerical values behind the chart and check whether they supported my observations.

**Q3. Commuting or recreation?**
Is there evidence people use the Burke-Gilman to *commute*, not just for fun? Compare **weekdays vs. weekends** (we made a `weekday` column for you), and maybe morning vs. evening. Define what a "commuting pattern" would look like in the data, then say whether you see it.

*Hint: `df.groupby("weekday")[["bikes","peds"]].sum()` gets you started. Weekends and weekdays will look different if commuting is real.*

In [ ]:
# Your code for Q3
# Show total counts for each day of the week
weekday_totals = df.groupby("weekday")[["bikes", "peds"]].sum()
print(weekday_totals)

# Separate weekdays and weekends using the weekday column
weekend_data = df[df["weekday"].isin(["Saturday", "Sunday"])]
weekday_data = df[
    df["weekday"].isin(
        ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
    )
]

# Calculate average bike and pedestrian counts by hour
weekend_hourly = (
    weekend_data.groupby("hour")[["bikes", "peds"]].mean()
)
weekday_hourly = (
    weekday_data.groupby("hour")[["bikes", "peds"]].mean()
)

# Create four charts with the same axes for comparison
fig, axes = plt.subplots(
    2, 2, figsize=(12, 8), sharex=True, sharey=True
)

charts = [
    (axes[0, 0], weekend_hourly["bikes"], "Bikes — Weekend", "blue"),
    (axes[0, 1], weekday_hourly["bikes"], "Bikes — Weekday", "blue"),
    (axes[1, 0], weekend_hourly["peds"], "Pedestrians — Weekend", "orange"),
    (axes[1, 1], weekday_hourly["peds"], "Pedestrians — Weekday", "orange"),
]

for ax, counts, title, color in charts:
    ax.plot(counts.index, counts.values, color=color)
    ax.set_title(title)
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Average passages per hour")
    ax.set_xticks(range(24))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    ax.tick_params(labelbottom=True)

plt.tight_layout()
plt.show()

**Your Q3 answer:**

I define a commuting pattern as morning and evening peaks on weekdays, without similar peaks on weekends. In the charts, both pedestrian and bicycle traffic show clear morning and evening peaks on weekdays, while weekend traffic peaks around midday. This suggests that the trail is used for both commuting and recreation, although the counts do not directly reveal trip purposes. I used the Q3 prompt documented in my AI statement to ask AI to help write code that produced four charts for comparison.

**Q4. Direction and flow.**
The counter records **north vs. south** separately. Are mornings mostly one direction and evenings the other? What would that tell you about who's using the trail and where they're heading?

In [ ]:
# Your code for Q4
import pandas as pd
import matplotlib.pyplot as plt

# Read the table
traffic = pd.read_csv("bg_traffic_bike_ped_2019.csv")
traffic.columns = traffic.columns.str.strip().str.lower()
traffic = traffic.rename(columns={
    "northbound bikes": "bike_north",
    "southbound bikes": "bike_south",
    "northbound pedestrians": "ped_north",
    "southbound pedestrians": "ped_south",
    "northbound pedestrains": "ped_north",
    "southbound pedestrains": "ped_south"
})
# Convert times such as "08:00:00" into hour numbers
traffic["hour"] = pd.to_datetime(
    traffic["hour of the day"], format="%H:%M:%S"
).dt.hour

# Convert directional counts to numbers
count_columns = ["bike_north", "bike_south", "ped_north", "ped_south"]
traffic[count_columns] = traffic[count_columns].apply(
    pd.to_numeric, errors="raise"
)

# Sort the rows by hour
traffic = traffic.sort_values("hour")

# Create two charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

axes[0].plot(
    traffic["hour"], traffic["bike_north"],
    color="blue", label="Northbound"
)
axes[0].plot(
    traffic["hour"], traffic["bike_south"],
    color="orange", label="Southbound"
)
axes[0].set_title("Bicycle traffic by direction")

axes[1].plot(
    traffic["hour"], traffic["ped_north"],
    color="blue", label="Northbound"
)
axes[1].plot(
    traffic["hour"], traffic["ped_south"],
    color="orange", label="Southbound"
)
axes[1].set_title("Pedestrian traffic by direction")

for ax in axes:
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Passage count")
    ax.set_xticks(range(24))
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Your Q4 answer:**

The chart shows that bicycle traffic is mainly northbound in the morning and southbound in the evening. This directional reversal is consistent with our commuting hypothesis, suggesting that some cyclists may travel north to work or school and return south toward home in the evening. In contrast, the pedestrian curves are similar in both directions, providing less clear evidence of directional commuting. However, the data does not show whether the morning and evening travelers are the same people, so we cannot confirm this explanation.I used AI to help write code to create the charts. I pointed out that the file contained northbound and southbound traffic counts and asked for charts comparing these counts by hour of the day for analysis.

## Part 3 — Make it reproducible

Good data work can be picked up and re-run by someone else. Two short deliverables:

**1. A `README.md`** in your `A1` folder. Use this template:

```
# A1 — Working with Open Data

**Dataset:** Burke-Gilman Trail counter north of NE 70th St
**Source:** https://data.seattle.gov/Transportation/.../2z5v-ecg8  (pulled YYYY-MM-DD)
**License / terms of use:** <find it on the dataset's page and note it here>

## Questions I asked
- ...

## How to re-run
Open `A1.ipynb` and run all cells (needs `requests`, `pandas`, `matplotlib`).
```

**2. A `LICENSE`** — keep the license file in this folder, and in your README **note the dataset's own license or terms of use** (open datasets almost always have one; find it on the dataset's page). Using open data responsibly means knowing what you're allowed to do with it — that's part of the assignment, not an afterthought.

### One more thing: read the data critically

In 3–4 sentences, name **one limitation or bias** in this dataset and how it could mislead someone who trusted it uncritically.

*(A sensor counts bodies, not intentions. It's one location on a long trail. Some hours may be missing when the counter was offline. A tandem bike counts once. What else?)* This is a warm-up for A2, where finding bias in data is the whole assignment.

**Your answer:**

One limitation of this dataset is that it records passages past the counter but does not record why people use the trail. Morning and evening peaks may reflect commuting, but they could also include exercise, walking, or other activities. Assuming that all peak-hour traffic comes from people traveling to work or school could overestimate the trail’s use for commuting. Therefore, patterns in timing and direction can support a commuting hypothesis but cannot directly prove trip purposes.

## Optional — going further

If you want to push further (not required for full credit):
- Which **day of the week** is busiest? Is it the same for bikes and pedestrians?
- Has the trail gotten **busier over time**? (the data goes back to 2014 — change the year in the query)
- Do fewer people ride when it's **cold or wet**? (join with a Seattle [weather dataset](https://data.seattle.gov/))
- Compare with another counter — the [Fremont Bridge](https://data.seattle.gov/Transportation/Fremont-Bridge-Bicycle-Counter/65db-xm6k) or the [MTS Trail](https://data.seattle.gov/Transportation/MTS-Trail-west-of-I-90-Bridge-Bicycle-and-Pedestri/u38e-ybnc).

## A note on using AI

You may use an LLM (ChatGPT, Claude, …) to help you write code or make a plot. Treat it as a **tool you supervise, not an author you trust**:
- paste the **prompt** you used into a Markdown cell,
- **run** the output and check it against what you expected,
- note **what you changed and why**.

Here, AI is your assistant. In **A3** it becomes your subject of study. Code that runs but that you can't explain won't earn credit — the point is that *you* can ask a question of data and defend the answer.

Q1: I am analyzing 2019 hourly traffic data. Each row represents one hourly record. I loaded and prepared the dataset using this code:

import requests
import pandas as pd
import matplotlib.pyplot as plt

api_endpoint = "https://data.seattle.gov/resource/2z5v-ecg8.json?"
api_parameters = "$limit=50000&$where=date > '2019-01-01T00:00:00' AND date < '2020-01-01T00:00:00'"

response = requests.get(api_endpoint + api_parameters)
raw_data = response.json()
df = pd.DataFrame(raw_data)

count_cols = [
    "bike_north", "bike_south", "ped_north", "ped_south",
    "bgt_north_of_ne_70th_total"
]
df[count_cols] = df[count_cols].apply(pd.to_numeric, errors="coerce")

df["date"] = pd.to_datetime(df["date"])
df["hour"] = df["date"].dt.hour
df["weekday"] = df["date"].dt.day_name()
df["bikes"] = df["bike_north"] + df["bike_south"]
df["peds"] = df["ped_north"] + df["ped_south"]

Please write code to find the hour of the day (0–23) with the highest combined bicycle and pedestrian traffic, summed across all records in 2019. Print the busiest hour and its combined total, and briefly explain how the code works. I want the busiest recurring hour of the day, rather than a single dated hourly record.

Q2: Please show the numerical values behind the hourly bicycle and pedestrian traffic chart so I can check whether they support my observations.

Q3: Please write code to create four charts showing weekend bicycle counts, weekday bicycle counts, weekend pedestrian counts, and weekday pedestrian counts. The x-axis of each chart should be the hour of the day. First, distinguish weekday and weekend data using the information in the table produced by df.groupby("weekday")[["bikes", "peds"]].sum(), then select the appropriate data to create the charts.

Q4: Please help me create two charts: one for pedestrians and one for bicycles. In each chart, use different colored lines for northbound and southbound traffic so the two directions can be compared. Use the data from bg_traffic_bike_ped_2019.csv, which contains the columns northbound bikes, southbound bikes, northbound pedestrains, and southbound pedestrains.
(I modified because some wrong variable names.)